# Aspect control through the melt season

How strongly aspect paces runoff onset depends on *when* a band melts. Per elevation band, the band's own aspect
profile (onset minus the band's median across aspects) is fitted with a cosine; its amplitude A is half the band's
sunny–shady difference (2A). Two questions:

1. **The seasonal arch.** Grouping a range's bands by the month in which they usually melt, how does the band
   amplitude change through the season?
2. **Does an early year sharpen the contrast?** When a band melts earlier than usual, is its amplitude larger, and does
   that depend on how far the band's usual melt date is from the equinox? Tested against a null that shuffles the
   amplitude anomalies among a band's years.

```
reads (results/<version>/)                         writes
──────────────────────────                         ──────
mountain_range_band_aspect.csv  ──┐                 results/<version>/aspect_seasonality_arch.csv      range x melt month
mountain_range_statistics.csv  ───┴──► this ───►    results/<version>/aspect_seasonality_early_melt.csv  the test, by class
                                                    figures/<version>/aspect_seasonal_arch.png
```

Both tables are written by `range_statistics.ipynb`; run it first. The test is per range (a slope within each range,
then counts and medians across ranges, never one regression over pooled band-years): pooled, a few marginal-snow ranges
with band dates that swing by 50–80 days dominate the fit. It is reported for the included ranges and, as a check, for
every range with at least `MIN_YEARS` years of band values.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gsro_analysis import paths, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
UNIT = 'mountain_ranges'
results_dir = paths.resultsdir(UNIT, VERSION)
figdir = paths.figdir(UNIT, VERSION)

MIN_YEARS = 9                      # a range enters the early-melt test with band values in at least this many years
EQUINOX_DOWY = 172                 # day of water year of the spring equinox (1 Oct start in the north, 1 Apr in the south)
EQUINOX_CLASSES = [(-90, -30, '> 30 d before the equinox'), (-30, 30, 'within 30 d of the equinox'),
                   (30, 60, '30-60 d after'), (60, 90, '60-90 d after'), (90, 200, '> 90 d after')]
N_PERM = 1000
SEED = 0
EXEMPLARS = ['Hindu Kush', 'Himalaya', 'Sierra Nevada', 'European Alps', 'Brooks Range', 'Southern Alps']
MONTH_STARTS = np.array([1, 32, 62, 93, 124, 152, 183, 213, 244, 274, 305, 336])   # day of water year
MONTHS_NH = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
MONTHS_SH = ['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar']

In [ ]:
band_aspect_df = pd.read_csv(results_dir / 'mountain_range_band_aspect.csv', comment=None)
band_aspect_df = band_aspect_df[[c for c in band_aspect_df.columns if not c.startswith('_')]]
band_aspect_df

In [ ]:
statistics_df = pd.read_csv(results_dir / 'mountain_range_statistics.csv').set_index('name')
included = statistics_df.index[statistics_df['included']]
print(f'{len(included)} included ranges')

## 1. Bands: their usual melt date, amplitude and anomalies

In [ ]:
median_df = band_aspect_df[band_aspect_df['water_year'] == 'median'].drop(columns='water_year')
median_df = median_df.rename(columns={'onset_dowy': 'onset_median', 'amp_d': 'amp_median', 'earliest_aspect_deg': 'az_median'})
year_df = band_aspect_df[band_aspect_df['water_year'] != 'median'].astype({'water_year': int})
bands_df = year_df.merge(median_df[['name', 'elevation', 'onset_median', 'amp_median']], on=['name', 'elevation'])
bands_df = bands_df.dropna(subset=['onset_dowy', 'amp_d', 'onset_median', 'amp_median'])
bands_df['included'] = bands_df['name'].isin(included)
bands_df['d_date'] = bands_df['onset_dowy'] - bands_df['onset_median']          # days later (+) or earlier (-) than usual
bands_df['d_sunny_shady'] = 2 * (bands_df['amp_d'] - bands_df['amp_median'])     # change of the band's sunny-shady difference
bands_df['latitude'] = bands_df['name'].map(statistics_df['centroid_latitude'])
bands_df['from_equinox'] = bands_df['onset_median'] - EQUINOX_DOWY
print(f"{len(bands_df)} band-years in {bands_df['name'].nunique()} ranges")
bands_df.head()

## 2. The seasonal arch

Per range, the pixel-weighted mean band sunny–shady difference (2A) of the bands that usually melt in each month
(water-year months, so the same column is the same season in both hemispheres).

In [ ]:
arch_src = median_df[median_df['name'].isin(included)].dropna(subset=['amp_median']).copy()
arch_src['month_index'] = np.searchsorted(MONTH_STARTS, arch_src['onset_median'], side='right') - 1
arch_df = (arch_src.groupby(['name', 'month_index'])
           .apply(lambda g: pd.Series({'sunny_shady_d': 2 * np.average(g['amp_median'], weights=g['median_pixels']),
                                       'n_bands': len(g), 'pixels': g['median_pixels'].sum()}), include_groups=False)
           .reset_index())
arch_df['pixel_share'] = arch_df['pixels'] / arch_df.groupby('name')['pixels'].transform('sum')
south = arch_df['name'].map(statistics_df['centroid_latitude']) < 0
arch_df['month'] = np.where(south, np.array(MONTHS_SH)[arch_df['month_index']], np.array(MONTHS_NH)[arch_df['month_index']])
arch_df.assign(**results.provenance(config)).to_csv(results_dir / 'aspect_seasonality_arch.csv', index=False, float_format='%.4g')
arch_df[arch_df['name'].isin(EXEMPLARS)].pivot(index='name', columns='month_index', values='sunny_shady_d').round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for name in [n for n in EXEMPLARS if n in arch_df['name'].values]:
    g = arch_df[(arch_df['name'] == name) & (arch_df['pixel_share'] >= 0.01)]
    ax.plot(g['month_index'], g['sunny_shady_d'], marker='o', label=name)
ax.set_xticks(range(12)); ax.set_xticklabels([f'{a}\n({b})' for a, b in zip(MONTHS_NH, MONTHS_SH)], fontsize=7)
ax.set_ylabel('band sunny–shady difference (2A, days)'); ax.set_xlabel('month the band usually melts (southern hemisphere in brackets)')
ax.legend(fontsize=7, frameon=False); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(figdir / 'aspect_seasonal_arch.png', dpi=300)

## 3. Does an early year sharpen the contrast?

Per range and per class of bands (by the band's usual melt date relative to the equinox): the regression slope of the
change in the band's sunny–shady difference on the change in its melt date over that range's band-years, reported as
**days of sunny–shady difference per 10 days earlier** (positive = an early year has the larger contrast); a range enters
a class with at least `MIN_BAND_YEARS` band-years in it. Across ranges: the median slope and the share of ranges with a
positive slope. Null: the amplitude anomalies are shuffled among each band's own years (keeps each band's variance and
its date anomalies), `N_PERM` times, and the same two numbers recomputed.

In [ ]:
counts = bands_df.groupby('name')['water_year'].nunique()
test_df = bands_df[bands_df['name'].isin(counts.index[counts >= MIN_YEARS])].copy()
test_df['equinox_class'] = None
for lo, hi, label in EQUINOX_CLASSES:
    test_df.loc[(test_df['from_equinox'] >= lo) & (test_df['from_equinox'] < hi), 'equinox_class'] = label
test_df = test_df.dropna(subset=['equinox_class']).reset_index(drop=True)
print(f"{len(test_df)} band-years, {test_df['name'].nunique()} ranges")
test_df.groupby('equinox_class', sort=False).size()

In [ ]:
MIN_BAND_YEARS = 20
labels = [label for _, _, label in EQUINOX_CLASSES]

def per_range_slopes(df):
    # {class: array of per-range slopes, days of 2A per 10 days earlier}
    out = {}
    for label in labels:
        g = df[df['equinox_class'] == label]
        out[label] = np.array([-10 * np.polyfit(r['d_date'], r['d_sunny_shady'], 1)[0]
                               for _, r in g.groupby('name') if len(r) >= MIN_BAND_YEARS and r['d_date'].std() > 0])
    return out

tables = []
for selection, sel_df in (('included ranges', test_df[test_df['included']]), ('all ranges', test_df)):
    sel_df = sel_df.reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    observed = per_range_slopes(sel_df)
    groups = sel_df.groupby(['name', 'elevation']).indices
    null_median, null_share = [], []
    for _ in range(N_PERM):
        shuffled = sel_df['d_sunny_shady'].to_numpy().copy()
        for idx in groups.values():
            shuffled[idx] = rng.permutation(shuffled[idx])
        nul = per_range_slopes(sel_df.assign(d_sunny_shady=shuffled))
        null_median.append([np.median(nul[l]) if len(nul[l]) else np.nan for l in labels])
        null_share.append([(nul[l] > 0).mean() if len(nul[l]) else np.nan for l in labels])
    null_median, null_share = np.array(null_median), np.array(null_share)
    rows = []
    for j, label in enumerate(labels):
        sl = observed[label]
        med, share = (np.median(sl), (sl > 0).mean()) if len(sl) else (np.nan, np.nan)
        rows.append({'equinox_class': label, 'selection': selection, 'ranges': len(sl),
                     'median_d_per_10d_earlier': med, 'share_positive': share,
                     'null_median_p2.5': np.nanpercentile(null_median[:, j], 2.5), 'null_median_p97.5': np.nanpercentile(null_median[:, j], 97.5),
                     'null_share_p2.5': np.nanpercentile(null_share[:, j], 2.5), 'null_share_p97.5': np.nanpercentile(null_share[:, j], 97.5),
                     'p_median_two_sided': float((np.abs(null_median[:, j] - np.nanmedian(null_median[:, j])) >= abs(med - np.nanmedian(null_median[:, j]))).mean())})
    tables.append(pd.DataFrame(rows))
early_melt_df = pd.concat(tables).set_index('equinox_class')
early_melt_df.assign(**results.provenance(config)).to_csv(results_dir / 'aspect_seasonality_early_melt.csv', float_format='%.4g')
early_melt_df.round(3)